# DavidAU Qwen GGUF vision

Corrected version of the supplied [Colab inference demo](https://colab.research.google.com/notebook#fileId=https%3A//huggingface.co/DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF.ipynb). Uses the exact IQ2_M GGUF plus `mmproj-F16.gguf` and the upstream `MTMDChatHandler`. This runs the existing model; it does not create or train a new model.

Code/schema validated locally. GPU execution and image quality remain unverified. Requires a current CUDA-enabled llama-cpp-python build supporting this architecture; installation compiles native code and downloads multi-GB weights in Colab.

## 1. Build the CUDA backend

In [ ]:
import os, subprocess, sys
subprocess.run(['nvidia-smi'], check=True)
os.environ['CMAKE_ARGS'] = '-DGGML_CUDA=on'
os.environ['FORCE_CMAKE'] = '1'
subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--no-cache-dir', 'git+https://github.com/abetlen/llama-cpp-python.git', 'huggingface_hub'], check=True)
from llama_cpp.llama_chat_format import MTMDChatHandler
print('MTMD handler available')

## 2. Load the model and projector, then describe an image
Use a plain HTTPS URL, not Markdown link syntax. Replace the default image with a chart image or a Colab-local file path.

In [ ]:
from pathlib import Path
Path('/content/davidau_gguf_vision.py').write_text('#!/usr/bin/env python3\n"""Run the requested DavidAU IQ2_M model with its required vision projector."""\nimport argparse\nfrom pathlib import Path\n\nREPO = "DavidAU/Qwen3.8-27B-TURBO-Fable-Cold-Fusion-735-882-Heretic-Uncensored-NEO-CODER-MAX-MTP-GGUF"\nREVISION = "a51791d22b62b03aa5132feaac27147f32f289f7"\nGGUF = "Qwen3.8-27B-TurboFCFusion-735-882-Here-Uncen-NEO-CODER-MAX-IQ2_M.gguf"\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--image", default="https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg")\n    parser.add_argument("--prompt", default="Describe this image in one sentence.")\n    parser.add_argument("--gpu-layers", type=int, default=-1)\n    args = parser.parse_args()\n    from huggingface_hub import hf_hub_download\n    from llama_cpp import Llama\n    try:\n        from llama_cpp.llama_chat_format import MTMDChatHandler\n    except ImportError as e:\n        raise RuntimeError("Install a current llama-cpp-python build exposing MTMDChatHandler before downloading model weights") from e\n    model = hf_hub_download(REPO, GGUF, revision=REVISION)\n    projector = hf_hub_download(REPO, "mmproj-F16.gguf", revision=REVISION)\n    image_url = args.image\n    if not image_url.startswith(("https://", "http://", "data:")):\n        import base64\n        import mimetypes\n        path = Path(image_url)\n        mime = mimetypes.guess_type(path.name)[0] or "image/png"\n        image_url = f"data:{mime};base64," + base64.b64encode(path.read_bytes()).decode()\n    handler = MTMDChatHandler(clip_model_path=projector)\n    with Llama(model_path=model, chat_handler=handler, n_ctx=4096, n_gpu_layers=args.gpu_layers, verbose=False) as llm:\n        result = llm.create_chat_completion(messages=[{"role": "user", "content": [\n            {"type": "text", "text": args.prompt}, {"type": "image_url", "image_url": {"url": image_url}}]}],\n            max_tokens=256, temperature=0.2)\n        print(result["choices"][0]["message"]["content"])\n\n\nif __name__ == "__main__":\n    main()\n')
IMAGE = 'https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg'
PROMPT = 'Describe this image in one sentence.'
subprocess.run([sys.executable, '/content/davidau_gguf_vision.py', '--image', IMAGE, '--prompt', PROMPT], check=True)

## 3. Record the environment

In [ ]:
Path('/content/gguf-environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))